# **Start Section:**


In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d02_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # pile_settlement train
    shutil.copy(glob.glob("/kaggle/input/**/d02_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # pile_settlement test
    print("Kaggle: pile_settlement data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: pile_settlement data copied to ./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel.
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9 mapie==0.6.0 puncc==0.8.0
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
import ngboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from sklearn.model_selection import KFold
from PIL import Image as PImage
from openpyxl.utils.dataframe import dataframe_to_rows
from pytorch_tabnet import TabNetRegressor
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


In [4]:
# Go to find & replace button and replace (pile_settlement_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.


In [5]:
feature_names = ['D', 'L', 'N30_Ava', 'Cu_Ava', 'N30_Base', 'Cu_Base', 'UL', 'Q ']
TARGET_LABEL = 'Settlement (S_exp)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)


In [6]:
train_data_path = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")


Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))



Shape of training data: (371, 9)
First 5 rows of training data:
      D     L    N30_Ava      Cu_Ava  N30_Base  Cu_Base  UL       Q      S_exp 
0  1.0  43.5   3.317241  105.408046        50      0.0   1     0.00   0.000000
1  0.8  24.0   8.875000   17.708333         0     50.0   1   675.00   0.691250
2  0.8  24.0   5.032258   20.000000         0     40.0   1  1012.50   0.870417
3  1.2  70.0  18.214286   14.142857        50      0.0   2  7485.15  19.161300
4  1.0  34.0   6.741176   89.117647        50      0.0   2  3007.59   6.552300

Shape of test data: (186, 9)
First 5 rows of test data:
      D     L    N30_Ava     Cu_Ava  N30_Base  Cu_Base  UL        Q     S_exp 
0  0.8  24.0   8.875000  17.708333         0     50.0   2  1687.500  3.295000
1  1.0  38.0  26.631579  49.539474        50      0.0   2  2250.000  4.595960
2  0.8  24.0   5.032258  20.000000         0     40.0   1  1687.500  1.721250
3  1.0  35.5  28.400000  36.676056        50      0.0   1   741.259  0.441964
4  1.2  57.0

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)



Shape of X_train: (371, 8)
Shape of y_train: (371,)
Shape of X_test: (186, 8)
Shape of y_test: (186,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])



First five rows of normalized X_train:
[[ 0.11501862  0.7026739  -1.39285389  2.03926221  0.47278688 -0.36847246
  -0.88040878 -0.93211799]
 [-1.33640677 -0.87840793 -0.78031847 -0.77862641 -1.76108898  0.85111254
  -0.88040878 -0.72895693]
 [-1.33640677 -0.87840793 -1.2038373  -0.70499262 -1.76108898  0.60719554
  -0.88040878 -0.6273764 ]
 [ 1.566444    2.85132356  0.24898912 -0.89318909  0.47278688 -0.36847246
   1.13583602  1.32075756]
 [ 0.11501862 -0.06759674 -1.0154929   1.5158337   0.47278688 -0.36847246
   1.13583602 -0.02689552]]

First five rows of normalized X_test:
[[-1.33640677 -0.87840793 -0.78031847 -0.77862641 -1.76108898  0.85111254
   1.13583602 -0.42421534]
 [ 0.11501862  0.25672774  1.1766814   0.2441433   0.47278688 -0.36847246
   1.13583602 -0.25491446]
 [-1.33640677 -0.87840793 -1.2038373  -0.70499262 -1.76108898  0.60719554
  -0.88040878 -0.42421534]
 [ 0.11501862  0.05402494  1.37158379 -0.16917252  0.47278688 -0.36847246
  -0.88040878 -0.70901434]
 [ 1.566444

# **Functions:**


In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']


In [12]:
# FIX M4: one miscoverage level for the conformal notebooks. AlphaNet LEARNS its own level alpha~,
# so ALPHA is only the nominal reference value reported next to it.
ALPHA = 0.1


# FIX A1: lambda is the size/coverage trade-off chosen by the user (Gauthier, Bach & Jordan 2026,
# eq. 9). The official code reports every lambda separately and never compares losses across
# lambdas. This table evaluates, for EVERY lambda, its best run (lowest final training loss within
# that lambda) on the test set, so the trade-off can be seen.
def _acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test, y_test):
    rows = []
    for lam, res in sorted(all_results.items(), key=lambda kv: kv[0]):
        final_losses = res["all_losses"][:, -1]
        r = int(np.argmin(final_losses))
        ev = evaluate_acp(model=model, alpha_net=res["models"][r], X_calib=X_calib, y_calib=y_calib,
                          X_test=X_test, y_test=y_test)
        a = ev.get("alpha_hat", ev.get("alpha"))
        rows.append({
            "lambda": lam,
            "best_run (1-based)": r + 1,
            "final_train_loss": float(final_losses[r]),
            "alpha_learned": a,
            "Target_coverage": 1.0 - a,
            "test_coverage": ev["coverage"],
            "width": ev["interval_width"],
            "Miss_over_alpha": (1.0 - ev["coverage"]) / a,
        })
    return pd.DataFrame(rows)


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _to_numpy_2d(x):
    arr = x.to_numpy() if hasattr(x, "to_numpy") else np.asarray(x)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    return arr.astype(np.float32)

def _to_numpy_1d(y):
    arr = y.to_numpy() if hasattr(y, "to_numpy") else np.asarray(y)
    return arr.reshape(-1).astype(np.float32)

def _ensure_excel_workbook(excel_file_path):
    if not excel_file_path:
        return
    os.makedirs(os.path.dirname(excel_file_path), exist_ok=True)
    if not os.path.exists(excel_file_path):
        wb = Workbook()
        wb.active.title = "Init"
        wb.save(_ensure_parent_dir(excel_file_path))

def save_plot_to_excel(fig, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    with io.BytesIO() as buf:
        fig.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        img = Image(PImage.open(buf))
        workbook = load_workbook(_ensure_excel_file(excel_file_path))
        base_name = sheet_name
        i = 1
        while sheet_name in workbook.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        worksheet = workbook.create_sheet(title=sheet_name)
        worksheet.add_image(img, "A1")
        workbook.save(_ensure_parent_dir(excel_file_path))

def save_values_to_excel(results_dict, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    df = pd.DataFrame(results_dict)
    workbook = load_workbook(_ensure_excel_file(excel_file_path))
    base_name = sheet_name
    i = 1
    while sheet_name in workbook.sheetnames:
        sheet_name = f"{base_name}_{i}"
        i += 1
    worksheet = workbook.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            worksheet.cell(row=r_idx, column=c_idx, value=value)
    workbook.save(_ensure_parent_dir(excel_file_path))

def save_dataframe_to_excel(df, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    workbook = load_workbook(_ensure_excel_file(excel_file_path))
    base_name = sheet_name
    i = 1
    while sheet_name in workbook.sheetnames:
        sheet_name = f"{base_name}_{i}"
        i += 1
    worksheet = workbook.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            worksheet.cell(row=r_idx, column=c_idx, value=value)
    workbook.save(_ensure_parent_dir(excel_file_path))

class AlphaNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
        # FIX A2: start the output near alpha~ = 1 (sigmoid(5) ~ 0.993), as in the official code
        # (github.com/GauthierE/adaptive-coverage-policies, initialize_to_one), so alpha~ starts in the
        # region alpha~ > 1/(n+1) where the width formula is valid. Was: random initial output.
        with torch.no_grad():
            self.net[2].weight.data.normal_(0, 0.01)
            self.net[2].bias.data.fill_(5.0)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def interval_size(scores, alpha, eps=1e-6):
    n = scores.numel()
    denominator = torch.clamp(alpha * (n + 1) - 1.0, min=eps)
    return 2.0 * scores.sum() / denominator

def _compute_scores_without_index(base_model, X_calib, y_calib, holdout_idx):
    loo_X = np.delete(X_calib, holdout_idx, axis=0)
    loo_y = np.delete(y_calib, holdout_idx, axis=0)
    return np.abs(base_model.predict(loo_X) - loo_y)

def _build_loo_feature_matrix(base_model, X_calib, y_calib):
    rows = []
    for idx in range(len(X_calib)):
        scores = _compute_scores_without_index(base_model, X_calib, y_calib, idx)
        rows.append([scores.sum()])
    return torch.tensor(np.asarray(rows, dtype=np.float32))

def _resolve_model_params(model_class, best_params):
    params = dict(best_params or {})
    if "TabNet" in model_class.__name__ and "verbose" in params:
        params.pop("verbose", None)
    try:
        model_class(**params)
        return params
    except Exception as e:
        # FIX B5: was a silent fallback to default parameters
        print(f"[Warning] {model_class.__name__}: tuned parameters rejected ({type(e).__name__}: {e}); "
              "falling back to the model's DEFAULT parameters")
        return {}

def _train_alpha_net(
    base_model,
    X_calib,
    y_calib,
    lambdas=(10, 20, 50),
    num_runs=3,
    epochs=50,
    batch_size=32,
    learning_rate=1e-2,  # FIX A3 (was 1e-3)
    min_steps=2000,  # FIX A3
    alpha_clip=1e-3,
    seed=42,
):
    X_train_alpha = _build_loo_feature_matrix(base_model, X_calib, y_calib)
    # FIX A2: the network input (sum of leave-one-out errors) is divided by its mean, so inputs are ~1 on
    # every dataset (raw sums can be in the thousands, where the sigmoid is flat and training stalls).
    # The same number divides the test feature in evaluate_acp (stored on each net as feature_scale_).
    feature_scale = float(X_train_alpha.mean())
    feature_scale = feature_scale if feature_scale > 0 else 1.0
    X_train_alpha = X_train_alpha / feature_scale
    # FIX A3: leave-one-out score vectors computed once (identical values; was one predict() per row per step)
    loo_scores = [torch.tensor(_compute_scores_without_index(base_model, X_calib, y_calib, i), dtype=torch.float32, device=DEVICE)
                  for i in range(len(X_calib))]
    all_results = {}
    for lambda_reg in lambdas:
        lambda_losses = []
        lambda_sizes = []
        lambda_alphas = []
        lambda_models = []
        for run_idx in range(num_runs):
            local_seed = seed + run_idx
            torch.manual_seed(local_seed)
            np.random.seed(local_seed)
            random.seed(local_seed)
            dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
            loader = DataLoader(dataset, batch_size=min(batch_size, len(X_train_alpha)), shuffle=True)
            alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(DEVICE)
            optimizer = optim.Adam(alpha_net.parameters(), lr=learning_rate)
            all_losses = []
            all_sizes = []
            all_alphas = []
            # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
            # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
            # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
            # now matches the exact minimiser of the same loss for every lambda.
            n_epochs = max(epochs, int(np.ceil(min_steps / len(loader))))
            for _ in range(n_epochs):
                epoch_losses = []
                epoch_sizes = []
                epoch_alphas = []
                alpha_net.train()
                for x_batch, idx_batch in loader:
                    x_batch = x_batch.to(DEVICE)
                    alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)
                    batch_sizes = []
                    for j, idx in enumerate(idx_batch.tolist()):
                        batch_sizes.append(interval_size(loo_scores[idx], alpha_pred[j]))  # FIX A3: cached
                    batch_sizes = torch.stack(batch_sizes)
                    loss = (batch_sizes + lambda_reg * alpha_pred).mean()
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    epoch_losses.append(float(loss.item()))
                    epoch_sizes.append(float(batch_sizes.mean().item()))
                    epoch_alphas.append(float(alpha_pred.mean().item()))
                all_losses.append(float(np.mean(epoch_losses)))
                all_sizes.append(float(np.mean(epoch_sizes)))
                all_alphas.append(float(np.mean(epoch_alphas)))
            alpha_net.eval()
            lambda_losses.append(np.asarray(all_losses, dtype=np.float32))
            lambda_sizes.append(np.asarray(all_sizes, dtype=np.float32))
            lambda_alphas.append(np.asarray(all_alphas, dtype=np.float32))
            alpha_net.feature_scale_ = feature_scale  # FIX A2
            lambda_models.append(alpha_net)
        all_results[lambda_reg] = {
            "all_losses": np.vstack(lambda_losses),
            "all_sizes": np.vstack(lambda_sizes),
            "all_alphas": np.vstack(lambda_alphas),
            "models": lambda_models,
        }
    return all_results

def _moving_average(values, window=10):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    if window <= 1 or arr.size < window:
        return arr.copy()
    kernel = np.ones(window, dtype=float) / float(window)
    return np.convolve(arr, kernel, mode="valid")

def _smooth_runs(run_matrix, window=10):
    smoothed_runs = [_moving_average(run_curve, window=window) for run_curve in np.asarray(run_matrix)]
    if not smoothed_runs:
        return np.array([]), np.array([])
    min_len = min(len(curve) for curve in smoothed_runs)
    if min_len == 0:
        return np.array([]), np.array([])
    stacked = np.asarray([curve[:min_len] for curve in smoothed_runs], dtype=float)
    return stacked.mean(axis=0), stacked.std(axis=0)

def plot_acp_training_dynamics(all_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]
    for idx, (lambda_reg, result_dict) in enumerate(sorted(all_results.items(), key=lambda item: item[0])):
        color = colors[idx % len(colors)]
        loss_mean, loss_std = _smooth_runs(result_dict["all_losses"], window=10)
        size_mean, size_std = _smooth_runs(result_dict["all_sizes"], window=10)
        alpha_mean, alpha_std = _smooth_runs(result_dict["all_alphas"], window=10)
        if loss_mean.size == 0:
            continue
        epochs_axis = np.arange(1, len(loss_mean) + 1)
        axes[0].plot(epochs_axis, loss_mean, color=color, label=f"lambda={lambda_reg}")
        axes[0].fill_between(epochs_axis, loss_mean - loss_std, loss_mean + loss_std, color=color, alpha=0.2)
        axes[1].plot(epochs_axis, size_mean, color=color, label=f"lambda={lambda_reg}")
        axes[1].fill_between(epochs_axis, size_mean - size_std, size_mean + size_std, color=color, alpha=0.2)
        axes[2].plot(epochs_axis, alpha_mean, color=color, label=f"lambda={lambda_reg}")
        axes[2].fill_between(epochs_axis, alpha_mean - alpha_std, alpha_mean + alpha_std, color=color, alpha=0.2)
    axes[0].set_title("Smoothed Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[1].set_title("Smoothed Mean Size")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Mean Size")
    axes[2].set_title(r"Smoothed Mean $\tilde{\alpha}$")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel(r"Mean $\tilde{\alpha}$")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(loc="best", fontsize="small")
    plt.tight_layout()
    return fig

def _select_best_alpha_net(all_results, lambda_main=50):
    # FIX A1: lambda is the user's size/coverage trade-off; the best RUN is chosen within lambda_main by its
    # final training loss. Was: final losses compared ACROSS lambdas; since the loss contains lambda * alpha~,
    # the smallest lambda (10, the narrowest / lowest-coverage setting) always won.
    if lambda_main not in all_results:
        raise ValueError(f"lambda_main={lambda_main} not in trained lambdas {sorted(all_results)}")
    final_losses = all_results[lambda_main]["all_losses"][:, -1]
    run_idx = int(np.argmin(final_losses))
    return lambda_main, run_idx, all_results[lambda_main]["models"][run_idx]


def evaluate_acp(model, alpha_net, X_calib, y_calib, X_test, y_test):
    X_calib_np = _to_numpy_2d(X_calib)
    y_calib_np = _to_numpy_1d(y_calib)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)
    calib_scores = np.abs(model.predict(X_calib_np) - y_calib_np)
    test_feature = torch.tensor([[calib_scores.sum() / getattr(alpha_net, "feature_scale_", 1.0)]], dtype=torch.float32, device=DEVICE)  # FIX A2
    with torch.no_grad():
        alpha_hat = float(torch.clamp(alpha_net(test_feature), min=1e-3, max=1.0 - 1e-3).item())
    score_tensor = torch.tensor(calib_scores, dtype=torch.float32, device=DEVICE)
    interval_width = float(interval_size(score_tensor, torch.tensor(alpha_hat, device=DEVICE)).item())
    y_pred = np.asarray(model.predict(X_test_np)).reshape(-1)
    lower = y_pred - 0.5 * interval_width
    upper = y_pred + 0.5 * interval_width
    coverage = float(np.mean((y_test_np >= lower) & (y_test_np <= upper)))
    rmse = float(np.sqrt(mean_squared_error(y_test_np, y_pred)))
    mae = float(np.mean(np.abs(y_test_np - y_pred)))
    return {
        "X_test": X_test_np,
        "y_test": y_test_np,
        "y_pred": y_pred,
        "lower": lower,
        "upper": upper,
        "alpha": alpha_hat,
        "interval_width": interval_width,
        "coverage": coverage,
        "rmse": rmse,
        "mae": mae,
        # FIX A1: the guarantee is about the LEARNED level (eq. 3: E[miss / alpha~] <= 1), so report
        # coverage against 1 - alpha~ and the miss rate divided by alpha~.
        "target_coverage": 1.0 - alpha_hat,
        "miss_over_alpha": (1.0 - coverage) / alpha_hat,
    }

def conformal_predictions_ACP(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path,
    alpha=ALPHA,  # FIX M4: was 0.1 (nominal only; AlphaNet learns alpha~)
    lambdas=(10, 20, 50),
    lambda_main=50,  # FIX A1: the lambda used for the main result (paper reports lambda = 50)
    num_runs=3,
    epochs=50,
):
    X_train_np = _to_numpy_2d(X_train)
    y_train_np = _to_numpy_1d(y_train)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)
    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train_np,
        y_train_np,
        test_size=0.35,
        random_state=42,
    )
    safe_params = _resolve_model_params(model_class, best_params)
    model = model_class(**safe_params)
    model.fit(X_fit, y_fit)
    all_results = _train_alpha_net(
        base_model=model,
        X_calib=X_calib,
        y_calib=y_calib,
        lambdas=lambdas,
        num_runs=num_runs,
        epochs=epochs,
    )
    best_lambda, best_run_idx, alpha_net = _select_best_alpha_net(all_results, lambda_main)  # FIX A1
    eval_bundle = evaluate_acp(
        model=model,
        alpha_net=alpha_net,
        X_calib=X_calib,
        y_calib=y_calib,
        X_test=X_test_np,
        y_test=y_test_np,
    )
    y_true = eval_bundle["y_test"]
    y_pred = eval_bundle["y_pred"]
    lower = eval_bundle["lower"]
    upper = eval_bundle["upper"]
    print(f"{model_name} ACP best lambda: {best_lambda}")
    print(f"{model_name} ACP best run: {best_run_idx + 1}")
    print(f"{model_name} ACP alpha: {eval_bundle['alpha']:.4f}")
    print(f"{model_name} ACP width: {eval_bundle['interval_width']:.4f}")
    print(f"{model_name} ACP coverage: {eval_bundle['coverage'] * 100:.2f}%")
    print(f"{model_name} ACP RMSE: {eval_bundle['rmse']:.4f}")
    print(f"{model_name} ACP MAE: {eval_bundle['mae']:.4f}")
    print(f"{model_name} ACP target coverage (1 - alpha~): {eval_bundle['target_coverage'] * 100:.2f}%  |  miss / alpha~: {eval_bundle['miss_over_alpha']:.3f}  # FIX A1")
    fig_pred, axs = plt.subplots(2, 1, figsize=(11, 12))
    idx = np.arange(len(y_true))
    axs[0].scatter(idx, y_true, label="True", color="blue", s=12, alpha=0.7)
    axs[0].fill_between(idx, lower, upper, color="gray", alpha=0.45, label="ACP Interval")
    axs[0].plot(idx, y_pred, color="red", linewidth=1.2, label="Prediction")
    axs[0].set_title(f"ACP Intervals - {model_name}")
    axs[0].set_xlabel("Sample Number")
    axs[0].set_ylabel(f"{TARGET_LABEL}")
    axs[0].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
    abs_err = np.abs(y_true - y_pred)
    widths = upper - lower
    axs[1].scatter(widths, abs_err, s=14, alpha=0.7, color="teal", label="Samples")
    axs[1].axvline(np.mean(widths), linestyle="--", color="gray", label="Mean Width")
    axs[1].axhline(np.mean(abs_err), linestyle="--", color="orange", label="Mean Absolute Error")
    axs[1].set_title(f"ACP Width vs Absolute Error - {model_name}")
    axs[1].set_xlabel("Interval Width")
    axs[1].set_ylabel("Absolute Error")
    axs[1].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
    plt.tight_layout()
    fig_dynamics = plot_acp_training_dynamics(all_results)
    if excel_file_path:
        save_plot_to_excel(fig_pred, excel_file_path, "conformal_predictions_ACP")
        save_plot_to_excel(fig_dynamics, excel_file_path, "acp_training_dynamics")
        save_values_to_excel({
                "ACP_true": y_true,
                "ACP_pred": y_pred,
                "ACP_lower": lower,
                "ACP_upper": upper,
                "ACP_alpha": np.full(len(y_true), eval_bundle["alpha"]),
                "ACP_interval_width": np.full(len(y_true), eval_bundle["interval_width"]),
                "ACP_is_covered": ((y_true >= lower) & (y_true <= upper)).astype(int),
                "ACP_best_lambda": np.full(len(y_true), best_lambda),
                "ACP_best_run": np.full(len(y_true), best_run_idx + 1),
                "ACP_target_coverage": np.full(len(y_true), eval_bundle["target_coverage"]),  # FIX A1
                "ACP_miss_over_alpha": np.full(len(y_true), eval_bundle["miss_over_alpha"]),  # FIX A1
            },
            excel_file_path,
            "conformal_predictions_ACP_values",
        )
        training_summary_rows = []
        training_curve_rows = []
        for lambda_reg, result_dict in sorted(all_results.items(), key=lambda item: item[0]):
            final_losses = result_dict["all_losses"][:, -1]
            final_sizes = result_dict["all_sizes"][:, -1]
            final_alphas = result_dict["all_alphas"][:, -1]
            training_summary_rows.append(
                {
                    "lambda": lambda_reg,
                    "runs": result_dict["all_losses"].shape[0],
                    "best_run": int(np.argmin(final_losses)) + 1,
                    "final_loss_mean": float(np.mean(final_losses)),
                    "final_loss_std": float(np.std(final_losses)),
                    "final_size_mean": float(np.mean(final_sizes)),
                    "final_size_std": float(np.std(final_sizes)),
                    "final_alpha_mean": float(np.mean(final_alphas)),
                    "final_alpha_std": float(np.std(final_alphas)),
                }
            )
            mean_loss_curve = result_dict["all_losses"].mean(axis=0)
            mean_size_curve = result_dict["all_sizes"].mean(axis=0)
            mean_alpha_curve = result_dict["all_alphas"].mean(axis=0)
            for epoch_idx in range(len(mean_loss_curve)):
                training_curve_rows.append(
                    {
                        "lambda": lambda_reg,
                        "epoch": epoch_idx + 1,
                        "mean_loss": float(mean_loss_curve[epoch_idx]),
                        "mean_size": float(mean_size_curve[epoch_idx]),
                        "mean_alpha": float(mean_alpha_curve[epoch_idx]),
                    }
                )
        save_dataframe_to_excel(pd.DataFrame(training_summary_rows), excel_file_path, "acp_training_summary")
        save_dataframe_to_excel(pd.DataFrame(training_curve_rows), excel_file_path, "acp_training_curves")
        save_dataframe_to_excel(_acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test_np, y_test_np),
                                excel_file_path, "acp_lambda_comparison")  # FIX A1
    plt.close(fig_pred)
    plt.close(fig_dynamics)
    eval_bundle["model"] = model
    eval_bundle["alpha_net"] = alpha_net
    eval_bundle["all_results"] = all_results
    eval_bundle["best_lambda"] = best_lambda
    eval_bundle["best_run"] = best_run_idx + 1
    return eval_bundle


In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

def _train_alpha_net_multi(
    base_model,
    X_calib,
    y_calib,
    X_train_alpha,
    lambdas=(10, 20, 50),
    num_runs=3,
    epochs=50,
    batch_size=32,
    lr=1e-2,  # FIX A3 (was 1e-3)
    alpha_clip=1e-3,
    min_steps=2000,  # FIX A3
    seed=42,
):
    # FIX A2: the network input (sum of leave-one-out errors) is divided by its mean, so inputs are ~1 on
    # every dataset (raw sums can be in the thousands, where the sigmoid is flat and training stalls).
    # The same number divides the test feature in evaluate_acp (stored on each net as feature_scale_).
    feature_scale = float(X_train_alpha.mean())
    feature_scale = feature_scale if feature_scale > 0 else 1.0
    X_train_alpha = X_train_alpha / feature_scale
    # FIX A3: leave-one-out score vectors computed once (identical values; was one predict() per row per step)
    loo_scores = [torch.tensor(_compute_scores_without_index(base_model, X_calib, y_calib, i), dtype=torch.float32, device=DEVICE)
                  for i in range(len(X_calib))]
    Dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
    loader = DataLoader(Dataset, batch_size=min(batch_size, len(X_train_alpha)), shuffle=True)

    all_results = {}
    for lambda_reg in lambdas:
        lambda_losses = []
        lambda_sizes = []
        lambda_alphas = []
        lambda_models = []

        for run_idx in range(num_runs):
            torch.manual_seed(seed + run_idx)
            np.random.seed(seed + run_idx)

            alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(DEVICE)
            optimizer = optim.Adam(alpha_net.parameters(), lr=lr)

            run_losses = []
            run_sizes = []
            run_alphas = []

            # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
            # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
            # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
            # now matches the exact minimiser of the same loss for every lambda.
            n_epochs = max(epochs, int(np.ceil(min_steps / len(loader))))
            for _ in range(n_epochs):
                alpha_net.train()
                epoch_losses = []
                epoch_sizes = []
                epoch_alphas = []

                for x_batch, idx_batch in loader:
                    x_batch = x_batch.to(DEVICE)
                    alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)

                    batch_sizes = []
                    for j, idx in enumerate(idx_batch.tolist()):
                        batch_sizes.append(interval_size(loo_scores[idx], alpha_pred[j]))  # FIX A3: cached

                    batch_sizes = torch.stack(batch_sizes)
                    loss = (batch_sizes + float(lambda_reg) * alpha_pred).mean()

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    epoch_losses.append(float(loss.item()))
                    epoch_sizes.append(float(batch_sizes.mean().item()))
                    epoch_alphas.append(float(alpha_pred.mean().item()))

                run_losses.append(float(np.mean(epoch_losses)))
                run_sizes.append(float(np.mean(epoch_sizes)))
                run_alphas.append(float(np.mean(epoch_alphas)))

            alpha_net.eval()
            lambda_losses.append(run_losses)
            lambda_sizes.append(run_sizes)
            lambda_alphas.append(run_alphas)
            alpha_net.feature_scale_ = feature_scale  # FIX A2
            lambda_models.append(alpha_net)

        all_results[float(lambda_reg)] = {
            "all_losses": np.asarray(lambda_losses, dtype=np.float32),
            "all_sizes": np.asarray(lambda_sizes, dtype=np.float32),
            "all_alphas": np.asarray(lambda_alphas, dtype=np.float32),
            "models": lambda_models,
        }

    return all_results


def _moving_average(values, window=10):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values
    window = max(1, min(int(window), values.size))
    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(values, kernel, mode="valid")


def _smooth_runs(run_matrix, window=10):
    run_matrix = np.asarray(run_matrix, dtype=np.float32)
    if run_matrix.ndim == 1:
        run_matrix = run_matrix.reshape(1, -1)
    if run_matrix.size == 0:
        return np.asarray([]), np.asarray([])

    smoothed_runs = [_moving_average(run, window=window) for run in run_matrix]
    min_len = min(len(curve) for curve in smoothed_runs)
    if min_len == 0:
        return np.asarray([]), np.asarray([])

    trimmed = np.vstack([curve[:min_len] for curve in smoothed_runs])
    return trimmed.mean(axis=0), trimmed.std(axis=0)


def plot_acp_training_dynamics(all_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if not all_results:
        for ax in axes:
            ax.text(0.5, 0.5, "No training results", ha="center", va="center")
            ax.axis("off")
        return fig

    sorted_results = sorted(all_results.items(), key=lambda item: item[0])
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(sorted_results)))

    for color, (lambda_reg, result) in zip(colors, sorted_results):
        loss_mean, loss_std = _smooth_runs(result["all_losses"], window=10)
        size_mean, size_std = _smooth_runs(result["all_sizes"], window=10)
        alpha_mean, alpha_std = _smooth_runs(result["all_alphas"], window=10)

        if loss_mean.size:
            x_loss = np.arange(1, loss_mean.size + 1)
            axes[0].plot(x_loss, loss_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[0].fill_between(x_loss, loss_mean - loss_std, loss_mean + loss_std, color=color, alpha=0.2)

        if size_mean.size:
            x_size = np.arange(1, size_mean.size + 1)
            axes[1].plot(x_size, size_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[1].fill_between(x_size, size_mean - size_std, size_mean + size_std, color=color, alpha=0.2)

        if alpha_mean.size:
            x_alpha = np.arange(1, alpha_mean.size + 1)
            axes[2].plot(x_alpha, alpha_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[2].fill_between(x_alpha, alpha_mean - alpha_std, alpha_mean + alpha_std, color=color, alpha=0.2)

    axes[0].set_title("Smoothed Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].set_title("Smoothed Mean Size")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Size")

    axes[2].set_title("Smoothed Mean Alpha-tilde")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Alpha-tilde")

    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize="small")

    plt.tight_layout()
    return fig


def _select_best_alpha_net(all_results, lambda_main=50):
    # FIX A1: lambda is the user's size/coverage trade-off; the best RUN is chosen within lambda_main by its
    # final training loss. Was: final losses compared ACROSS lambdas; since the loss contains lambda * alpha~,
    # the smallest lambda (10, the narrowest / lowest-coverage setting) always won.
    if lambda_main not in all_results:
        raise ValueError(f"lambda_main={lambda_main} not in trained lambdas {sorted(all_results)}")
    losses = all_results[lambda_main]["all_losses"]
    if losses.size == 0:
        raise RuntimeError("AlphaNet training produced no valid runs.")
    final_losses = losses[:, -1]
    run_idx = int(np.argmin(final_losses))
    return {
        "lambda": lambda_main,
        "run_idx": run_idx,
        "final_loss": float(final_losses[run_idx]),
        "model": all_results[lambda_main]["models"][run_idx],
    }


def evaluate_acp(model, alpha_net, X_calib, y_calib, X_test, y_test):
    X_calib_np = _to_numpy_2d(X_calib)
    y_calib_np = _to_numpy_1d(y_calib)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)

    calib_preds = np.asarray(model.predict(X_calib_np)).reshape(-1)
    calib_scores = np.abs(calib_preds - y_calib_np)

    feature = torch.tensor([[calib_scores.sum() / getattr(alpha_net, "feature_scale_", 1.0)]], dtype=torch.float32, device=DEVICE)  # FIX A2
    with torch.no_grad():
        alpha_hat = float(torch.clamp(alpha_net(feature), min=1e-3, max=1.0 - 1e-3).item())

    scores_tensor = torch.tensor(calib_scores, dtype=torch.float32, device=DEVICE)
    interval_width = float(
        interval_size(scores_tensor, torch.tensor(alpha_hat, dtype=torch.float32, device=DEVICE)).item()
    )

    y_pred = np.asarray(model.predict(X_test_np)).reshape(-1)
    lower = y_pred - 0.5 * interval_width
    upper = y_pred + 0.5 * interval_width

    coverage = float(np.mean((y_test_np >= lower) & (y_test_np <= upper)))
    rmse = float(np.sqrt(mean_squared_error(y_test_np, y_pred)))
    mae = float(np.mean(np.abs(y_test_np - y_pred)))

    return {
        "coverage": coverage,
        "interval_width": interval_width,
        "alpha_hat": alpha_hat,
        "rmse": rmse,
        "mae": mae,
        "y_pred": y_pred,
        "lower": lower,
        "upper": upper,
        "y_test": y_test_np,
        # FIX A1: the guarantee is about the LEARNED level (eq. 3: E[miss / alpha~] <= 1), so report
        # coverage against 1 - alpha~ and the miss rate divided by alpha~.
        "target_coverage": 1.0 - alpha_hat,
        "miss_over_alpha": (1.0 - coverage) / alpha_hat,
    }


def run_full_acp_pipeline(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path,
    lambdas=(10, 20, 50),
    lambda_main=50,  # FIX A1: the lambda used for the main result (paper reports lambda = 50)
    num_runs=3,
    epochs=50,
):
    X_train_np = _to_numpy_2d(X_train)
    y_train_np = _to_numpy_1d(y_train)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)

    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train_np, y_train_np, test_size=0.35, random_state=42
    )

    safe_params = _resolve_model_params(model_class, best_params)
    model = model_class(**safe_params)
    model.fit(X_fit, y_fit)

    X_train_alpha = _build_loo_feature_matrix(model, X_calib, y_calib)
    all_results = _train_alpha_net_multi(
        base_model=model,
        X_calib=X_calib,
        y_calib=y_calib,
        X_train_alpha=X_train_alpha,
        lambdas=lambdas,
        num_runs=num_runs,
        epochs=epochs,
    )

    dynamics_fig = plot_acp_training_dynamics(all_results)
    dynamics_fig.suptitle(f"ACP Training Dynamics - {model_name}")
    dynamics_fig.tight_layout(rect=[0, 0, 1, 0.95])
    if excel_file_path:
        save_plot_to_excel(dynamics_fig, excel_file_path, "acp_training_dynamics")
    plt.close(dynamics_fig)

    best_choice = _select_best_alpha_net(all_results, lambda_main)  # FIX A1
    metrics = evaluate_acp(
        model=model,
        alpha_net=best_choice["model"],
        X_calib=X_calib,
        y_calib=y_calib,
        X_test=X_test_np,
        y_test=y_test_np,
    )

    y_true = metrics["y_test"]
    y_pred = metrics["y_pred"]
    lower = metrics["lower"]
    upper = metrics["upper"]

    print(f"{model_name} best lambda: {best_choice['lambda']}")
    print(f"{model_name} best run: {best_choice['run_idx']}")
    print(f"{model_name} ACP alpha_hat: {metrics['alpha_hat']:.4f}")
    print(f"{model_name} ACP width: {metrics['interval_width']:.4f}")
    print(f"{model_name} ACP coverage: {metrics['coverage'] * 100:.2f}%")
    print(f"{model_name} ACP RMSE: {metrics['rmse']:.4f}")
    print(f"{model_name} ACP MAE: {metrics['mae']:.4f}")
    print(f"{model_name} ACP target coverage (1 - alpha~): {metrics['target_coverage'] * 100:.2f}%  |  miss / alpha~: {metrics['miss_over_alpha']:.3f}  # FIX A1")

    fig, axs = plt.subplots(2, 1, figsize=(11, 12))
    idx = np.arange(len(y_true))

    axs[0].scatter(idx, y_true, label="True", color="blue", s=12, alpha=0.7)
    axs[0].fill_between(idx, lower, upper, color="gray", alpha=0.45, label="ACP Interval")
    axs[0].plot(idx, y_pred, color="red", linewidth=1.2, label="Prediction")
    axs[0].set_title(f"ACP Intervals - {model_name}")
    axs[0].set_xlabel("Sample Number")
    axs[0].set_ylabel(f"{TARGET_LABEL}")
    axs[0].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)

    abs_err = np.abs(y_true - y_pred)
    widths = upper - lower
    axs[1].scatter(widths, abs_err, s=14, alpha=0.7, color="teal", label="Samples")
    axs[1].axvline(np.mean(widths), linestyle="--", color="gray", label="Mean Width")
    axs[1].axhline(np.mean(abs_err), linestyle="--", color="orange", label="Mean Absolute Error")
    axs[1].set_title(f"ACP Width vs Absolute Error - {model_name}")
    axs[1].set_xlabel("Interval Width")
    axs[1].set_ylabel("Absolute Error")
    axs[1].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)

    plt.tight_layout()

    if excel_file_path:
        save_plot_to_excel(fig, excel_file_path, "conformal_predictions_ACP")

        save_values_to_excel({
                "ACP_true": y_true,
                "ACP_pred": y_pred,
                "ACP_lower": lower,
                "ACP_upper": upper,
                "ACP_alpha": np.full(len(y_true), metrics["alpha_hat"]),
                "ACP_interval_width": np.full(len(y_true), metrics["interval_width"]),
                "ACP_is_covered": ((y_true >= lower) & (y_true <= upper)).astype(int),
                "ACP_target_coverage": np.full(len(y_true), metrics["target_coverage"]),  # FIX A1
                "ACP_miss_over_alpha": np.full(len(y_true), metrics["miss_over_alpha"]),  # FIX A1
            },
            excel_file_path,
            "conformal_predictions_ACP_values",
        )

        save_values_to_excel({
                "model_name": [model_name],
                "best_lambda": [best_choice["lambda"]],
                "best_run": [best_choice["run_idx"]],
                "best_final_loss": [best_choice["final_loss"]],
                "coverage": [metrics["coverage"]],
                "interval_width": [metrics["interval_width"]],
                "rmse": [metrics["rmse"]],
                "mae": [metrics["mae"]],
                "alpha_hat": [metrics["alpha_hat"]],
                "Target_coverage": [metrics["target_coverage"]],  # FIX A1
                "Miss_over_alpha": [metrics["miss_over_alpha"]],  # FIX A1
            },
            excel_file_path,
            "acp_final_metrics",
        )

        summary = {
            "lambda": [],
            "mean_final_loss": [],
            "std_final_loss": [],
            "mean_final_size": [],
            "std_final_size": [],
            "mean_final_alpha": [],
            "std_final_alpha": [],
        }
        curve_data = {
            "lambda": [],
            "run": [],
            "epoch": [],
            "loss": [],
            "size": [],
            "alpha_tilde": [],
        }

        for lambda_reg, result in sorted(all_results.items(), key=lambda item: item[0]):
            losses = result["all_losses"]
            sizes = result["all_sizes"]
            alphas = result["all_alphas"]

            summary["lambda"].append(lambda_reg)
            summary["mean_final_loss"].append(float(losses[:, -1].mean()))
            summary["std_final_loss"].append(float(losses[:, -1].std()))
            summary["mean_final_size"].append(float(sizes[:, -1].mean()))
            summary["std_final_size"].append(float(sizes[:, -1].std()))
            summary["mean_final_alpha"].append(float(alphas[:, -1].mean()))
            summary["std_final_alpha"].append(float(alphas[:, -1].std()))

            n_runs, n_epochs = losses.shape
            for run_idx in range(n_runs):
                for epoch_idx in range(n_epochs):
                    curve_data["lambda"].append(lambda_reg)
                    curve_data["run"].append(run_idx)
                    curve_data["epoch"].append(epoch_idx + 1)
                    curve_data["loss"].append(float(losses[run_idx, epoch_idx]))
                    curve_data["size"].append(float(sizes[run_idx, epoch_idx]))
                    curve_data["alpha_tilde"].append(float(alphas[run_idx, epoch_idx]))

        save_values_to_excel(summary, excel_file_path, "acp_training_summary")
        save_values_to_excel(curve_data, excel_file_path, "acp_training_curves")
        lambda_df = _acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test_np, y_test_np)  # FIX A1
        print(lambda_df.to_string(index=False))
        save_dataframe_to_excel(lambda_df, excel_file_path, "acp_lambda_comparison")

    plt.close(fig)

    return {
        "model": model,
        "all_results": all_results,
        "best_choice": best_choice,
        "metrics": metrics,
    }


In [15]:
# Unified ACP execution entry point:
# run_full_acp_pipeline(model_class, best_params, X_train, y_train, X_test, y_test, model_name, excel_file_path)


# **Tuned Parameters**


In [16]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/pile-settlement-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


# **Conformal Predictions with Lightgbm**


In [17]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')


In [18]:
run_full_acp_pipeline(
    LGBMRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "LightGBM",
    "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/LightGBM.xlsx",
)


LightGBM best lambda: 50
LightGBM best run: 2
LightGBM ACP alpha_hat: 0.3295
LightGBM ACP width: 16.9839
LightGBM ACP coverage: 82.80%
LightGBM ACP RMSE: 11.7456
LightGBM ACP MAE: 5.3254
LightGBM ACP target coverage (1 - alpha~): 67.05%  |  miss / alpha~: 0.522  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   3         14.836707       0.742146         0.257854       0.741935  7.443057         0.347728
   20.0                   3         21.033714       0.522118         0.477882       0.774194 10.626192         0.432481
   50.0                   3         33.411835       0.329528         0.670472       0.827957 16.983879         0.522089


{'model': LGBMRegressor(bagging_freq=1, colsample_bytree=0.7, learning_rate=0.05,
               max_depth=5, min_child_samples=1, min_child_weight=0.01,
               n_estimators=500, n_jobs=-1, num_leaves=63,
               objective='regression', random_state=42, reg_alpha=0,
               reg_lambda=1, subsample=0.8, verbose=-1),
 'all_results': {10.0: {'all_losses': array([[15.474886 , 15.409817 , 15.240123 , ..., 14.873006 , 14.870399 ,
           14.851109 ],
          [15.477634 , 15.442601 , 15.360652 , ..., 14.868758 , 14.833545 ,
           14.867826 ],
          [15.4750395, 15.432159 , 15.317153 , ..., 14.856849 , 14.867151 ,
           14.836707 ]], dtype=float32),
   'all_sizes': array([[5.5653114, 5.643406 , 5.9352503, ..., 7.364229 , 7.3796024,
           7.390528 ],
          [5.561323 , 5.604384 , 5.7274046, ..., 7.3609233, 7.3621416,
           7.429754 ],
          [5.564799 , 5.6226783, 5.797443 , ..., 7.3628573, 7.3887825,
           7.3732643]], dtype=float32

# **Conformal Predictions with XGBoost**


In [19]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')


In [20]:
run_full_acp_pipeline(
    XGBRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "XGBoost",
    "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/XGBoost.xlsx",
)


XGBoost best lambda: 50
XGBoost best run: 0
XGBoost ACP alpha_hat: 0.3590
XGBoost ACP width: 18.8561
XGBoost ACP coverage: 83.33%
XGBoost ACP RMSE: 11.9675
XGBoost ACP MAE: 5.4350
XGBoost ACP target coverage (1 - alpha~): 64.10%  |  miss / alpha~: 0.464  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         16.311634       0.813019         0.186981       0.731183  8.227328         0.330641
   20.0                   1         23.122536       0.568203         0.431797       0.790323 11.820420         0.369018
   50.0                   1         36.725166       0.359040         0.640960       0.833333 18.856148         0.464201


{'model': XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.5, colsample_bynode=None, colsample_bytree=0.5,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=0.5, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.15, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=-1,
              num_parallel_tree=None, ...),
 'all_results': {10.0: {'all_losses': array([[16.653828, 16.610483, 16.4972  , ..., 16.366861, 16.34727 ,
           16.311634],
          [16.657068, 16.631216, 16.57383 , ..., 16.35849 , 16.330517,
           16.365446],
          [16.6509

# **Conformal Predictions with GPBoost**


In [21]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')


In [22]:
run_full_acp_pipeline(
    GPBoostRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "GPBoost",
    "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/GPBoost.xlsx",
)


GPBoost best lambda: 50
GPBoost best run: 0
GPBoost ACP alpha_hat: 0.3610
GPBoost ACP width: 18.8036
GPBoost ACP coverage: 79.03%
GPBoost ACP RMSE: 11.6402
GPBoost ACP MAE: 5.8458
GPBoost ACP target coverage (1 - alpha~): 63.90%  |  miss / alpha~: 0.581  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         16.330515       0.809487         0.190513       0.672043  8.287548         0.405142
   20.0                   1         23.148628       0.571858         0.428142       0.752688 11.777933         0.432471
   50.0                   1         36.762245       0.361044         0.638956       0.790323 18.803619         0.580753


{'model': GPBoostRegressor(colsample_bytree=0.5, learning_rate=0.15, max_depth=3,
                  min_child_samples=1, n_estimators=400, num_leaves=63,
                  random_state=42, reg_alpha=0, reg_lambda=0.5, subsample=0.6,
                  verbose=-1),
 'all_results': {10.0: {'all_losses': array([[16.665997, 16.630775, 16.517517, ..., 16.389627, 16.374676,
           16.330515],
          [16.677004, 16.651665, 16.595802, ..., 16.382486, 16.350958,
           16.388521],
          [16.658281, 16.64071 , 16.564453, ..., 16.378435, 16.387096,
           16.357779]], dtype=float32),
   'all_sizes': array([[6.7563868, 6.8647375, 7.2171326, ..., 8.123255 , 8.165874 ,
           8.091325 ],
          [6.760693 , 6.813475 , 6.963175 , ..., 8.132665 , 8.127145 ,
           8.18605  ],
          [6.747954 , 6.8311596, 7.045873 , ..., 8.135046 , 8.1598425,
           8.131043 ]], dtype=float32),
   'all_alphas': array([[0.9909609 , 0.9766038 , 0.9300384 , ..., 0.8266373 , 0.82088023,


# **Conformal Predictions with NGBoost**


In [23]:
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')


In [24]:
run_full_acp_pipeline(
    NGBRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "NGBoost",
    "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/NGBoost.xlsx",
)


NGBoost best lambda: 50
NGBoost best run: 0
NGBoost ACP alpha_hat: 0.4241
NGBoost ACP width: 21.8068
NGBoost ACP coverage: 81.72%
NGBoost ACP RMSE: 15.0647
NGBoost ACP MAE: 7.2656
NGBoost ACP target coverage (1 - alpha~): 57.59%  |  miss / alpha~: 0.431  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         19.087278       0.958180         0.041820       0.715054  9.554358         0.297383
   20.0                   1         27.047195       0.676340         0.323660       0.768817 13.581241         0.341814
   50.0                   1         42.938084       0.424103         0.575897       0.817204 21.806816         0.431018


{'model': NGBRegressor(col_sample=0.5, learning_rate=0.05, minibatch_frac=0.9,
              n_estimators=1000,
              random_state=RandomState(MT19937) at 0x7EAA4A557C40, verbose=0),
 'all_results': {10.0: {'all_losses': array([[19.154041, 19.142426, 19.150322, ..., 19.147911, 19.1223  ,
           19.087278],
          [19.154268, 19.149757, 19.149452, ..., 19.122437, 19.081987,
           19.144947],
          [19.152197, 19.147886, 19.149248, ..., 19.12421 , 19.143969,
           19.10082 ]], dtype=float32),
   'all_sizes': array([[9.244484 , 9.377617 , 9.642876 , ..., 9.533387 , 9.528972 ,
           9.475592 ],
          [9.23796  , 9.311655 , 9.512805 , ..., 9.529336 , 9.48329  ,
           9.577637 ],
          [9.241955 , 9.339087 , 9.5882015, ..., 9.515697 , 9.548213 ,
           9.48769  ]], dtype=float32),
   'all_alphas': array([[0.99095565, 0.9764808 , 0.9507445 , ..., 0.9614523 , 0.95933264,
           0.96116877],
          [0.991631  , 0.9838103 , 0.9636649 , ..

# **Conformal Predictions with Gradient Boosting**


In [25]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


In [26]:
run_full_acp_pipeline(
    GradientBoostingRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "GradientBoosting",
    "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/Gradient Boosting.xlsx",
)


GradientBoosting best lambda: 50
GradientBoosting best run: 2
GradientBoosting ACP alpha_hat: 0.3506
GradientBoosting ACP width: 17.8368
GradientBoosting ACP coverage: 84.41%
GradientBoosting ACP RMSE: 11.2437
GradientBoosting ACP MAE: 5.2057
GradientBoosting ACP target coverage (1 - alpha~): 64.94%  |  miss / alpha~: 0.445  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   3         15.708856       0.786130         0.213870       0.725806  7.857257         0.348789
   20.0                   3         22.262989       0.555016         0.444984       0.768817 11.174725         0.416533
   50.0                   3         35.349289       0.350569         0.649431       0.844086 17.836752         0.444746


{'model': GradientBoostingRegressor(max_depth=10, max_features='sqrt',
                           min_impurity_decrease=0.1, min_samples_leaf=3,
                           min_samples_split=10, min_weight_fraction_leaf=0.01,
                           n_estimators=500, random_state=42),
 'all_results': {10.0: {'all_losses': array([[16.126625, 16.083914, 15.945749, ..., 15.730475, 15.724079,
           15.719315],
          [16.139639, 16.10171 , 16.042164, ..., 15.724756, 15.69016 ,
           15.727316],
          [16.126375, 16.100513, 16.005535, ..., 15.70707 , 15.726422,
           15.708856]], dtype=float32),
   'all_sizes': array([[6.2169914, 6.3177505, 6.643414 , ..., 7.799539 , 7.811446 ,
           7.8304067],
          [6.2233324, 6.2633786, 6.4092035, ..., 7.8181806, 7.787562 ,
           7.8506417],
          [6.216081 , 6.291061 , 6.486426 , ..., 7.800308 , 7.8329754,
           7.813735 ]], dtype=float32),
   'all_alphas': array([[0.99096346, 0.9766163 , 0.9302336 , ..., 

# **Conformal Predictions with CatBoost**


In [27]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')


In [28]:
run_full_acp_pipeline(
    CatBoostRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "CatBoost",
    "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/CatBoost.xlsx",
)


CatBoost best lambda: 50
CatBoost best run: 2
CatBoost ACP alpha_hat: 0.3456
CatBoost ACP width: 17.8903
CatBoost ACP coverage: 82.26%
CatBoost ACP RMSE: 11.1760
CatBoost ACP MAE: 5.2445
CatBoost ACP target coverage (1 - alpha~): 65.44%  |  miss / alpha~: 0.513  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         15.609523       0.785739         0.214261       0.725806  7.770475         0.348962
   20.0                   1         22.124571       0.547331         0.452669       0.790323 11.203038         0.383091
   50.0                   3         35.134216       0.345597         0.654403       0.822581 17.890259         0.513371


{'model': <catboost.core.CatBoostRegressor at 0x7eaa42ea3750>,
 'all_results': {10.0: {'all_losses': array([[16.061096 , 16.01165  , 15.868873 , ..., 15.634476 , 15.627755 ,
           15.609523 ],
          [16.066046 , 16.03342  , 15.96745  , ..., 15.6375265, 15.589509 ,
           15.638215 ],
          [16.066063 , 16.025816 , 15.9313545, ..., 15.619059 , 15.634775 ,
           15.613016 ]], dtype=float32),
   'all_sizes': array([[6.151507 , 6.2455072, 6.5662236, ..., 7.736188 , 7.754871 ,
           7.777271 ],
          [6.1497383, 6.1951594, 6.334421 , ..., 7.775494 , 7.7270055,
           7.809467 ],
          [6.155835 , 6.2162876, 6.412198 , ..., 7.7520447, 7.7834935,
           7.762551 ]], dtype=float32),
   'all_alphas': array([[0.990959  , 0.97661424, 0.93026483, ..., 0.78982884, 0.7872884 ,
           0.78322524],
          [0.9916307 , 0.98382604, 0.9633029 , ..., 0.78620315, 0.78625035,
           0.7828748 ],
          [0.9910228 , 0.98095286, 0.9519157 , ..., 0.78670

# **Conformal Predictions with HistGradientBoosting**


In [29]:
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')


In [30]:
run_full_acp_pipeline(
    HistGradientBoostingRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "HGBR",
    "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/HGBM.xlsx",
)


HGBR best lambda: 50
HGBR best run: 2
HGBR ACP alpha_hat: 0.4054
HGBR ACP width: 20.6813
HGBR ACP coverage: 82.26%
HGBR ACP RMSE: 13.6390
HGBR ACP MAE: 6.9873
HGBR ACP target coverage (1 - alpha~): 59.46%  |  miss / alpha~: 0.438  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   3         18.195431       0.912139         0.087861       0.645161  9.094449         0.389018
   20.0                   3         25.780107       0.643486         0.356514       0.725806 12.936932         0.426106
   50.0                   3         40.912746       0.405383         0.594617       0.822581 20.681328         0.437659


{'model': HistGradientBoostingRegressor(early_stopping=False, learning_rate=0.15,
                               max_bins=128, max_depth=7, max_iter=500,
                               max_leaf_nodes=63, min_samples_leaf=10,
                               random_state=42, validation_fraction=0.2),
 'all_results': {10.0: {'all_losses': array([[18.280184, 18.262035, 18.22668 , ..., 18.223673, 18.22018 ,
           18.222855],
          [18.283913, 18.272675, 18.25008 , ..., 18.210527, 18.170961,
           18.22834 ],
          [18.285233, 18.27099 , 18.231405, ..., 18.211437, 18.219368,
           18.19543 ]], dtype=float32),
   'all_sizes': array([[8.370608, 8.496599, 8.932505, ..., 9.088132, 9.065915, 9.069157],
          [8.367602, 8.434585, 8.619407, ..., 9.063303, 9.030578, 9.109313],
          [8.375008, 8.461826, 8.717157, ..., 9.066033, 9.064111, 9.057358]],
         dtype=float32),
   'all_alphas': array([[0.99095756, 0.9765437 , 0.9294176 , ..., 0.9135542 , 0.9154265 ,
       

# **Conformal Predictions with PGBM**


In [31]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [32]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())


Training on CPU


Estimator 0/300, Train metric: 22.1366
Estimator 1/300, Train metric: 20.8612
Estimator 2/300, Train metric: 19.7222
Estimator 3/300, Train metric: 18.3465
Estimator 4/300, Train metric: 17.2399
Estimator 5/300, Train metric: 16.3585
Estimator 6/300, Train metric: 15.5218
Estimator 7/300, Train metric: 14.6674
Estimator 8/300, Train metric: 14.0307


Estimator 9/300, Train metric: 13.4829
Estimator 10/300, Train metric: 12.9849
Estimator 11/300, Train metric: 12.5348
Estimator 12/300, Train metric: 12.0420
Estimator 13/300, Train metric: 11.6170
Estimator 14/300, Train metric: 11.1625
Estimator 15/300, Train metric: 10.7936
Estimator 16/300, Train metric: 10.4034
Estimator 17/300, Train metric: 10.0656


Estimator 18/300, Train metric: 9.7242
Estimator 19/300, Train metric: 9.4833
Estimator 20/300, Train metric: 9.1477
Estimator 21/300, Train metric: 8.9779
Estimator 22/300, Train metric: 8.7714
Estimator 23/300, Train metric: 8.5725
Estimator 24/300, Train metric: 8.3745


Estimator 25/300, Train metric: 8.1820
Estimator 26/300, Train metric: 8.0159
Estimator 27/300, Train metric: 7.8346
Estimator 28/300, Train metric: 7.6656
Estimator 29/300, Train metric: 7.4570
Estimator 30/300, Train metric: 7.3069
Estimator 31/300, Train metric: 7.1485


Estimator 32/300, Train metric: 6.9983
Estimator 33/300, Train metric: 6.8945
Estimator 34/300, Train metric: 6.7534
Estimator 35/300, Train metric: 6.6278
Estimator 36/300, Train metric: 6.5443
Estimator 37/300, Train metric: 6.4147


... (41 outputs left out)

Estimator 283/300, Train metric: 1.7971
Estimator 284/300, Train metric: 1.7951
Estimator 285/300, Train metric: 1.7929
Estimator 286/300, Train metric: 1.7863
Estimator 287/300, Train metric: 1.7833
Estimator 288/300, Train metric: 1.7811


Estimator 289/300, Train metric: 1.7799
Estimator 290/300, Train metric: 1.7770
Estimator 291/300, Train metric: 1.7717
Estimator 292/300, Train metric: 1.7697
Estimator 293/300, Train metric: 1.7691
Estimator 294/300, Train metric: 1.7663


Estimator 295/300, Train metric: 1.7634
Estimator 296/300, Train metric: 1.7625
Estimator 297/300, Train metric: 1.7595
Estimator 298/300, Train metric: 1.7573
Estimator 299/300, Train metric: 1.7558


        0.05        0.1       0.15        0.2        0.3        0.4  \
0   3.265267   3.274162   3.278327   3.282201   3.285167   3.289589   
1   2.858373   2.873633   2.881102   2.885765   2.895033   2.899088   
2   1.759889   1.765304   1.767196   1.770087   1.771839   1.774391   
3   0.728211   0.761958   0.777914   0.783758   0.797529   0.805729   
4  10.463576  10.919612  11.347294  11.522360  11.710670  11.904534   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0   3.293096   3.297314   3.300866   3.308504   3.310358   3.313658   
1   2.905445   2.912283   2.917895   2.925975   2.930824   2.937810   
2   1.775692   1.777804   1.779997   1.782661   1.784653   1.786698   
3   0.816341   0.823775   0.834706   0.846454   0.855265   0.866185   
4  12.014844  12.160096  12.234157  12.404488  12.610868  12.860250   

        0.95    Actual  
0   3.320782  3.295000  
1   2.942596  4.595960  
2   1.793037  1.721250  
3   0.888910  0.441964  
4  13.173440  0.00000

In [33]:
# ACP execution for PGBM
best_params = get_best_model_params(best_scores_autosampler, "PGBM")
run_full_acp_pipeline(
    model_class=PGBMWrapper,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/PGBM.xlsx",
)


Training on CPU
Estimator 0/300, Train metric: 23.0043
Estimator 1/300, Train metric: 21.8371
Estimator 2/300, Train metric: 20.8033
Estimator 3/300, Train metric: 20.0135
Estimator 4/300, Train metric: 19.3292
Estimator 5/300, Train metric: 18.5353
Estimator 6/300, Train metric: 17.8349
Estimator 7/300, Train metric: 17.1917
Estimator 8/300, Train metric: 16.6711
Estimator 9/300, Train metric: 16.1907
Estimator 10/300, Train metric: 15.6671
Estimator 11/300, Train metric: 15.2817
Estimator 12/300, Train metric: 14.8566


Estimator 13/300, Train metric: 14.4653
Estimator 14/300, Train metric: 13.9578


Estimator 15/300, Train metric: 13.7053
Estimator 16/300, Train metric: 13.2411
Estimator 17/300, Train metric: 12.9974
Estimator 18/300, Train metric: 12.6360
Estimator 19/300, Train metric: 12.3791
Estimator 20/300, Train metric: 12.0590
Estimator 21/300, Train metric: 11.8313
Estimator 22/300, Train metric: 11.5642
Estimator 23/300, Train metric: 11.3670


Estimator 24/300, Train metric: 11.0871
Estimator 25/300, Train metric: 10.9087


Estimator 26/300, Train metric: 10.7256
Estimator 27/300, Train metric: 10.4899
Estimator 28/300, Train metric: 10.2991
Estimator 29/300, Train metric: 10.1567
Estimator 30/300, Train metric: 9.9993
Estimator 31/300, Train metric: 9.8381
Estimator 32/300, Train metric: 9.6500


Estimator 33/300, Train metric: 9.4216


... (68 outputs left out)

Estimator 298/300, Train metric: 3.0482
Estimator 299/300, Train metric: 3.0465


PGBM best lambda: 50
PGBM best run: 2
PGBM ACP alpha_hat: 0.3527
PGBM ACP width: 18.1797
PGBM ACP coverage: 82.80%
PGBM ACP RMSE: 12.3012
PGBM ACP MAE: 5.9951
PGBM ACP target coverage (1 - alpha~): 64.73%  |  miss / alpha~: 0.488  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   3         15.896733       0.794197         0.205803       0.698925  7.974369         0.379094
   20.0                   3         22.529982       0.558581         0.441419       0.747312 11.384664         0.452375
   50.0                   3         35.774086       0.352652         0.647348       0.827957 18.179728         0.487854


{'model': PGBMWrapper(),
 'all_results': {10.0: {'all_losses': array([[16.284882, 16.24253 , 16.10994 , ..., 15.927672, 15.920297,
           15.90199 ],
          [16.29879 , 16.265352, 16.205421, ..., 15.91828 , 15.880592,
           15.920564],
          [16.285051, 16.261364, 16.168203, ..., 15.913932, 15.924279,
           15.896733]], dtype=float32),
   'all_sizes': array([[6.3752503, 6.4763207, 6.807299 , ..., 7.8984156, 7.9095755,
           7.908663 ],
          [6.3824873, 6.427094 , 6.572572 , ..., 7.9050016, 7.8833117,
           7.9485993],
          [6.374753 , 6.451947 , 6.649124 , ..., 7.9040155, 7.9282336,
           7.8979106]], dtype=float32),
   'all_alphas': array([[0.9909631 , 0.9766211 , 0.93026394, ..., 0.80292565, 0.8010722 ,
           0.79933274],
          [0.99163026, 0.9838257 , 0.9632849 , ..., 0.80132776, 0.799728  ,
           0.79719645],
          [0.9910299 , 0.9809416 , 0.9519079 , ..., 0.80099165, 0.79960454,
           0.7998823 ]], dtype=float32)

# **Conformal Predictions with TabNet**


In [34]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')
print(best_params)


{'n_d': 16, 'n_a': 32, 'n_steps': 10, 'gamma': 1.0, 'lambda_sparse': 0.001, 'optimizer_params': {'lr': 0.02}, 'mask_type': 'entmax', 'n_shared': 1, 'n_independent': 3, 'scheduler_params': {'step_size': 10, 'gamma': 0.9}, 'scheduler_fn': <class 'torch.optim.lr_scheduler.StepLR'>, 'seed': 42, 'verbose': 0}


In [35]:
class TabNetRegressorCP(TabNetRegressor):
    def fit(self, X, y, *args, **kwargs):
        y_arr = np.asarray(y)
        if y_arr.ndim == 1:
            y_arr = y_arr.reshape(-1, 1)
        return super().fit(X, y_arr, *args, **kwargs)

    def predict(self, X, *args, **kwargs):
        preds = super().predict(X, *args, **kwargs)
        return np.asarray(preds).reshape(-1)


best_params = get_best_model_params(best_scores_autosampler, "TabNet")
if isinstance(best_params, dict) and "verbose" in best_params:
    best_params = dict(best_params)
    best_params.pop("verbose", None)

run_full_acp_pipeline(
    TabNetRegressorCP,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "TabNet",
    "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/TabNet.xlsx",
)


epoch 0  | loss: 741.1001|  0:00:00s
epoch 1  | loss: 643.63019|  0:00:00s
epoch 2  | loss: 575.66443|  0:00:00s


epoch 3  | loss: 449.97375|  0:00:00s
epoch 4  | loss: 369.46793|  0:00:00s
epoch 5  | loss: 329.14224|  0:00:00s


epoch 6  | loss: 316.96307|  0:00:00s
epoch 7  | loss: 260.70425|  0:00:00s
epoch 8  | loss: 217.86299|  0:00:00s


epoch 9  | loss: 199.71552|  0:00:00s
epoch 10 | loss: 161.75125|  0:00:00s
epoch 11 | loss: 166.21378|  0:00:00s


epoch 12 | loss: 118.3615|  0:00:01s
epoch 13 | loss: 95.42853|  0:00:01s
epoch 14 | loss: 101.36607|  0:00:01s


epoch 15 | loss: 92.4231 |  0:00:01s
epoch 16 | loss: 77.42226|  0:00:01s
epoch 17 | loss: 61.30285|  0:00:01s


... (28 outputs left out)

epoch 99 | loss: 0.63785 |  0:00:08s


TabNet best lambda: 50
TabNet best run: 2
TabNet ACP alpha_hat: 0.3850
TabNet ACP width: 19.1640
TabNet ACP coverage: 86.02%
TabNet ACP RMSE: 13.6814
TabNet ACP MAE: 5.6991
TabNet ACP target coverage (1 - alpha~): 61.50%  |  miss / alpha~: 0.363  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   3         17.066849       0.855915         0.144085       0.731183  8.524240         0.314070
   20.0                   3         24.183573       0.604671         0.395329       0.801075 12.111397         0.328980
   50.0                   3         38.389275       0.384953         0.615047       0.860215 19.164032         0.363122


{'model': TabNetRegressorCP(n_d=16, n_a=32, n_steps=10, gamma=1.0, cat_idxs=[], cat_dims=[], cat_emb_dim=[], n_independent=3, n_shared=1, epsilon=1e-15, momentum=0.02, lambda_sparse=0.001, seed=42, clip_value=1, verbose=1, optimizer_fn=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.02}, scheduler_fn=<class 'torch.optim.lr_scheduler.StepLR'>, scheduler_params={'step_size': 10, 'gamma': 0.9}, mask_type='entmax', input_dim=8, output_dim=1, device_name='auto', n_shared_decoder=1, n_indep_decoder=1, grouped_features=[], compile_backend=''),
 'all_results': {10.0: {'all_losses': array([[17.27053 , 17.214233, 17.147532, ..., 17.094181, 17.089968,
           17.093622],
          [17.272388, 17.232956, 17.208488, ..., 17.089949, 17.091017,
           17.07314 ],
          [17.269802, 17.246077, 17.169636, ..., 17.068111, 17.094452,
           17.066849]], dtype=float32),
   'all_sizes': array([[7.3609614, 7.4476314, 7.8498454, ..., 8.502067 , 8.51918  ,
           8.498221 ],
     